# Momentum ranker — Nasdaq-100, standalone

**Nothing to clone.** Paste these cells into a blank Colab notebook and run them
top to bottom. The repository is private, so a `git clone` here would ask for a
token and fail; this notebook carries everything it needs instead.

It does two things, in order:

1. **Fetch tests** — the constituent list, then the prices. Each one prints what
   it got and what it could not get, so a failure tells you *which* fetch broke
   rather than that something did.
2. **The ranking** — 6-1 momentum across the constituents, with the same
   absolute filter the live book uses.

### What this is not

A second copy of the strategy. The repository's `cross_sectional_momentum` walks
the whole history and carries a **hysteresis band** — buy into the top 6, do not
sell until a name falls past rank 8 — which needs yesterday's holdings to apply
and is what keeps turnover down. One day in isolation has no yesterday, so the
band is not applied here.

The practical consequence: this shows **what a fresh book would buy today**, not
what the live book is holding. They agree on the strong names and disagree around
the edge, which is the band doing its job. For the book itself, run the repo.


## 1 · Setup


In [ ]:
# No clone, no token, no repo. Two packages and the standard scientific stack
# that Colab already has.
!pip install -q "yfinance>=0.2.40" "lxml>=4.9"

import pandas as pd, numpy as np, yfinance as yf

pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 120)
print("pandas", pd.__version__, "· yfinance", yf.__version__)


## 2 · Parameters

The live defaults, written out. Change them here and every cell below follows.


In [ ]:
LOOKBACK_MONTHS = 6      # the 6 in 6-1 momentum
SKIP_MONTHS     = 1      # the 1: the most recent month is skipped
N_HOLD          = 6      # names a fresh book would buy
MIN_HISTORY     = 260    # trading days a name needs before it can be ranked
SAFE_ASSET      = "BOXX" # the absolute filter's hurdle
START           = "2023-06-01"

# Months to trading days, the same rounding the repository uses.
LOOK = int(round(LOOKBACK_MONTHS * 21))
SKIP = int(round(SKIP_MONTHS * 21))
assert LOOK > SKIP, "the lookback has to be longer than the skip"
print(f"{LOOKBACK_MONTHS}-{SKIP_MONTHS} momentum = return from {LOOK} "
      f"sessions ago to {SKIP} sessions ago")


## 3 · Fetch test 1 — the constituents

Scraped live from Wikipedia. It **raises** rather than falling back to a
hardcoded list: a ranking that quietly used stale membership is worse than one
that stopped and said so. If the scrape breaks, set `TICKERS` by hand in the
next cell and carry on.


In [ ]:
def normalise(symbols):
    """Provider spellings turned into the one yfinance wants.

    Two substitutions, both to a dash, and each has cost a download:
    `.` is a share class (BRK.B -> BRK-B) and `/` is a preferred series
    (ORCL/PD -> ORCL-PD). Left alone the second 404s as "possibly delisted;
    no timezone found", which reads like a dead company rather than a
    misspelled symbol.
    """
    return (pd.Series(list(symbols), dtype="object").astype(str).str.strip()
            .str.upper()
            .str.replace(".", "-", regex=False)
            .str.replace("/", "-", regex=False))


def fetch_ndx(timeout: int = 20):
    """The current Nasdaq-100 constituents, newest membership.

    Looks for a table with a Ticker/Symbol column and at least 90 rows, rather
    than trusting a table index — Wikipedia reorders them.
    """
    tables = pd.read_html("https://en.wikipedia.org/wiki/Nasdaq-100")
    for t in tables:
        cols = {str(c).strip().lower() for c in t.columns}
        for key in ("ticker", "symbol"):
            if key in cols:
                col = [c for c in t.columns if str(c).strip().lower() == key][0]
                syms = normalise(t[col])
                syms = [s for s in syms if s.isascii() and 1 <= len(s) <= 6]
                if len(syms) >= 90:
                    return sorted(set(syms))
    raise RuntimeError("no constituents table with >=90 tickers — page changed")


try:
    TICKERS = fetch_ndx()
    print(f"FETCH 1 OK — {len(TICKERS)} constituents")
    print("  ", ", ".join(TICKERS[:12]), "...")
except Exception as exc:
    TICKERS = []
    print(f"FETCH 1 FAILED — {type(exc).__name__}: {exc}")
    print("   Set TICKERS = ['AAPL', 'MSFT', ...] yourself and re-run from here.")

# Today's membership applied to all history is SURVIVORSHIP BIAS: the names that
# dropped out are missing, and they are the ones that did badly. Fine for "what
# looks strong today", not fine for judging a backtest.


## 4 · Fetch test 2 — the prices

The one that actually breaks. It reports coverage, the last bar, and every
ticker that came back empty — so a thin download is visible rather than quietly
becoming a thin ranking.


In [ ]:
def fetch_closes(tickers, start=START, batch=100):
    """Adjusted closes for `tickers`, batched. `(frame, missing)`.

    A batch that fails is reported and skipped rather than aborting the run:
    losing five names out of a hundred is a slightly smaller sample, and losing
    the other ninety-five to them would be the wrong trade.
    """
    frames, missing = [], []
    for i in range(0, len(tickers), batch):
        chunk = tickers[i:i + batch]
        raw = yf.download(chunk, start=start, auto_adjust=True, progress=False,
                          actions=False, group_by="column", threads=True)
        if raw is None or raw.empty:
            missing += chunk
            continue
        # One ticker comes back flat, several come back as a MultiIndex.
        close = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw[["Close"]]
        if not isinstance(raw.columns, pd.MultiIndex):
            close.columns = chunk[:1]
        frames.append(close)

    if not frames:
        return pd.DataFrame(), list(tickers)
    px = pd.concat(frames, axis=1).sort_index()
    px.index = pd.to_datetime(px.index).tz_localize(None).normalize()
    empty = [c for c in px.columns if px[c].notna().sum() == 0]
    return px.drop(columns=empty), sorted(set(missing) | set(empty))


px, missing = fetch_closes(TICKERS + [SAFE_ASSET])
if px.empty:
    print("FETCH 2 FAILED — nothing came back at all.")
else:
    last = px.index.max()
    on_last = int(px.loc[last].notna().sum())
    print(f"FETCH 2 OK — {px.shape[1]} tickers x {len(px)} sessions")
    print(f"   last bar {last:%Y-%m-%d}, carried by {on_last} of {px.shape[1]} names")
    if missing:
        print(f"   no data for {len(missing)}: {', '.join(missing[:10])}")
    if SAFE_ASSET not in px.columns:
        print(f"   !! {SAFE_ASSET} is missing — the absolute filter cannot run")
    # A last bar only a handful of names carry is the provider mid-publish, not
    # a session. Ranking it would rank those few names and call it the universe.
    if on_last < 0.5 * px.shape[1]:
        print(f"   !! that bar is TORN ({on_last} names) — dropping it")
        px = px.iloc[:-1]


## 5 · The ranking

Two legs, both as the live ranker applies them:

* **6-1 momentum** — the return from `LOOK` sessions ago to `SKIP` sessions ago.
  Both ends are strictly in the past, so there is no look-ahead even at the last
  row. The recent month is skipped because short-horizon reversal works against
  momentum.
* **The absolute filter** — a name must also beat the safe asset over the *same*
  window. Without it the book buys the best of a falling market.


In [ ]:
def rank_today(px, safe_asset=SAFE_ASSET, look=LOOK, skip=SKIP,
               min_history=MIN_HISTORY):
    """One row per rankable name on the last bar, strongest first."""
    if px.empty or safe_asset not in px.columns:
        return pd.DataFrame()
    names = [c for c in px.columns if c != safe_asset]
    prices, safe = px[names], px[safe_asset]

    mom = prices.shift(skip) / prices.shift(look) - 1.0
    safe_mom = safe.shift(skip) / safe.shift(look) - 1.0
    history = prices.notna().cumsum()

    last = px.index.max()
    out = pd.DataFrame({
        "momentum": mom.loc[last],
        "history": history.loc[last],
    })
    out["beats_cash"] = out["momentum"] > safe_mom.loc[last]
    out["enough_history"] = out["history"] >= min_history
    out = out[out["momentum"].notna()]
    out = out.sort_values("momentum", ascending=False)
    # Ranked among the names that could be ranked at all, so a name excluded
    # for thin history does not silently occupy a slot in the numbering.
    ok = out["beats_cash"] & out["enough_history"]
    out["rank"] = np.where(ok, ok.cumsum(), np.nan)
    out.attrs["asof"] = last
    out.attrs["safe_momentum"] = float(safe_mom.loc[last])
    return out


table = rank_today(px)
if table.empty:
    print("Nothing could be ranked — check the two fetch tests above.")
else:
    asof = table.attrs["asof"]
    print(f"As of {asof:%Y-%m-%d} · {SAFE_ASSET} over the same window: "
          f"{table.attrs['safe_momentum']:+.2%}")
    show = table.head(20).copy()
    show["momentum"] = show["momentum"].map("{:+.1%}".format)
    show["rank"] = show["rank"].map(lambda r: "—" if r != r else f"{int(r)}")
    display(show[["rank", "momentum", "beats_cash", "enough_history"]])


## 6 · What a fresh book would buy

The top `N_HOLD` that clear both legs. **Not the live book** — no hysteresis
band, so no yesterday to hold on to. See the note at the top.


In [ ]:
picks = table[table["rank"].notna()].head(N_HOLD) if not table.empty else table
if picks.empty:
    print(f"Fully in cash — nothing beat {SAFE_ASSET} over the window.")
else:
    print(f"Top {len(picks)} on {table.attrs['asof']:%Y-%m-%d}, "
          f"equal weight {1 / len(picks):.1%} each:\n")
    for i, (sym, row) in enumerate(picks.iterrows(), 1):
        print(f"  {i}. {sym:<6} {row['momentum']:+7.1%}")

    blocked = table[(~table["beats_cash"]) & (table["momentum"] > 0)]
    print(f"\n{len(blocked)} names rose over the window and still lost to "
          f"{SAFE_ASSET} — the absolute filter is what keeps them out.")


## 7 · Take it with you


In [ ]:
out = table.copy()
out.index.name = "ticker"
fname = f"momentum_rank_{table.attrs['asof']:%Y%m%d}.csv" if not table.empty \
        else "momentum_rank_empty.csv"
out.to_csv(fname)
print("wrote", fname)

try:                       # Colab only; a no-op anywhere else
    from google.colab import files
    files.download(fname)
except Exception:
    pass


---

Every figure above comes from prices fetched in this notebook. The rules are the
live ones, but the engine is not: no hysteresis band, no rebalance calendar, no
slot weighting, and the universe is today's membership applied backwards. Good
for checking that a fetch works and seeing what is strong right now; not a
backtest, and not the book.
